In [ ]:
# Practice Problems Day 4

### Author:

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**Reading:** Geron, chapter 2, the section on **Create a Test Set**, and chapter 3, the section on **Measuring Accuracy Using Cross-Validation**

GOALS:

1. Get comfortable with `cross_val_score`, `GridSearchCV` and pipelines.
2. Learn to spot leakage, which is the point of the whole day.
3. See why stratification matters when the thing you care about is rare.

**How to turn this in.** Your repository on GitHub **is** your submission. There is no Pull Request to open any more.

Manage your git however you like. Use branches if you want them, or commit straight to `main` if you do not. What I need is only this:

1. The finished work is on your **`main`** branch.
2. It is **pushed to GitHub** before the deadline.
3. Your name is on the **Author** line at the top of this notebook.

```bash
git add .
git commit -m "Day 4 homework"
git push
```

If you did the work on a branch, merge it into `main` and push before the deadline:

```bash
git checkout main
git merge my-branch-name
git push
```

I grade from whatever is on GitHub at the deadline. If it is not pushed, I cannot see it.

This is part of **HW 2, due Sunday 9/13 at 11:59pm**, along with the Day 3 problems and **Weekly Homework 2**.

---

## Problem 1: Cross validation by hand

Use the same synthetic data from the Day 4 notes (the parabola with noise).

**1a.** Build a pipeline with `PolynomialFeatures(degree=8)`, `StandardScaler` and
`Ridge(alpha=1)`. Run `cross_val_score` with `cv=5` and print all five fold scores.

**1b.** Now run it again with `cv=10`. Print the ten scores.

**1c.** Compare the mean and the standard deviation for `cv=5` and `cv=10`. Which
gives the larger mean? Which gives the larger spread? Explain why the spread changes
in the direction it does.

**1d.** How many times was the model fit in total across both runs?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge

rng = np.random.default_rng(seed=42)
m = 200
X = 6 * rng.random((m, 1)) - 3
y = (0.5 * X**2 + X + 2 + rng.standard_normal((m, 1))).ravel()

# your code here

*Your answers here.*

---

## Problem 2: Find the leak

Each of the five snippets below has a leak, or does not. For each one, say
**leak or no leak**, and if it leaks, say exactly what information got where it
should not have.

No code needed. Write sentences.

**2a.**
```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

**2b.**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y)
model = make_pipeline(StandardScaler(), Ridge())
scores = cross_val_score(model, X_train, y_train, cv=5)
```

**2c.** You are predicting which students will drop a course. One feature is the
number of assignments they submitted during the whole semester.

**2d.**
```python
# 200 songs by 20 artists, predicting whether a song is a hit
X_train, X_test, y_train, y_test = train_test_split(songs, hits, test_size=0.25)
```

**2e.**
```python
X_train, X_test, y_train, y_test = train_test_split(X, y)
best_k = 0
for k in [5, 10, 20, 50]:
    sel = SelectKBest(k=k).fit(X_train, y_train)
    score = model.fit(sel.transform(X_train), y_train).score(sel.transform(X_test), y_test)
    if score > best_k: best_k = k     # pick k using the test set
```

*Your answers here.*

---

## Problem 3: Build the leak yourself

Reproduce the noise experiment from the notes, but smaller, and see where the leak
disappears.

**3a.** Make a dataset of 100 samples and **200** features of pure noise, with coin
flip labels. Confirm there is no signal.

**3b.** Select the best 10 features using all the data, then cross validate. What
accuracy do you get?

**3c.** Do it properly with the selection inside a pipeline. What accuracy now?

**3d.** The notes used 5000 features and got 0.850 the wrong way. You used 200.
Is your fake accuracy bigger or smaller than the notes got? Explain the pattern in
one sentence: what makes the leak worse?

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# your code here

*Your answers here.*

---

## Problem 4: Stratification

**4a.** Make a classification dataset with 300 samples where only **6 percent** are
positive. Give the positive class a real signal so a model can learn something.

**4b.** Split it with `KFold(n_splits=5, shuffle=True)` and print how many positives
land in each test fold. Try at least three different `random_state` values.

**4c.** Do the same with `StratifiedKFold`. What changes?

**4d.** Did any of your `KFold` runs produce a fold with **zero** positives? If yes,
say what score a model that answers "negative" for every single case would get on that
fold, and why that is a problem. If no, explain how you could make it more likely.

In [ ]:
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.datasets import make_classification

# your code here

*Your answers here.*

---

## Problem 5: A judgment call

No code. You are reviewing a colleague's work.

They built a model to predict which loan applicants will default. They report **94
percent accuracy** from 5-fold cross validation. Their pipeline scales the features
inside the pipeline, correctly. About **3 percent** of applicants in the data
actually defaulted.

**5a.** Before asking anything else, what is the one number you would want to compare
that 94 percent against? Compute it from what you have been told.

**5b.** Is 94 percent good? Defend your answer.

**5c.** They used `KFold`, not `StratifiedKFold`. Why does that matter more here than
it would on a balanced dataset?

**5d.** Name one further thing you would ask them for before you believed the model
was useful.

*There is more than one reasonable answer. I care about your reasoning.*

*Your answers here.*